# Questão 3 com dispersão e mapa de calor ligados

A pergunta é a da Questão 3 da primeira entrega, os orientadores são especialistas ou generalistas, agora com uma segunda parte: em quais temas atuam os orientadores de cada perfil?

Na primeira entrega, a dispersão mostrava quanto cada orientador se concentra em um tema e o mapa de calor mostrava quais temas eram esses, mas as duas visões eram separadas e o mapa tinha 93 colunas pequenas. Aqui elas estão ligadas, e o que for selecionado na dispersão aparece sozinho no mapa.

Para usar, arraste uma caixa sobre os pontos da dispersão. Um duplo clique no gráfico limpa a seleção.

## Dados

Usamos as 1046 bancas dos três programas com árvore de temas (PPG-CCMC, PIPGES e PPG-Mat), ligadas ao tema atribuído pelo modelo de linguagem. Tiramos as 41 bancas classificadas como "Outros", e ficam 1005 bancas de 136 orientadores. Por padrão só entram orientadores com pelo menos 4 bancas, o que dá 93.

## Interações

Já existiam na primeira entrega, em gráficos separados, o hover, o filtro por programa e por nível (tema ou macro-tema), o mínimo de bancas, a busca por nome e a ordem das colunas do mapa. Aqui eles controlam as duas visões ao mesmo tempo.

São novos o brushing, que é a seleção por caixa na dispersão, e o linking, que redesenha o mapa só com os orientadores selecionados. O duplo clique limpa a seleção.

## Decisões de design

- A caixa é sempre diagonal, porque a seleção padrão aceita faixas só horizontais ou só verticais, que misturam perfis.
- O mapa mantém a ordem escolhida e mostra só os temas que algum selecionado usou.
- Os nomes do eixo do mapa são cortados em 24 caracteres para economizar espaço, e o nome completo aparece no hover.
- A dispersão e o mapa ficam um sobre o outro, porque lado a lado o mapa ficava apertado.

In [ ]:
# Setup: imports, caminhos e paleta (Okabe-Ito, a mesma da parte 1)
import unicodedata
import zlib

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

CAMINHO = "../../data/eventos.csv"                           # dataset
ARQ_TEMAS = "../parte1/question3/temas_llm.csv"         # tema de cada banca (gerado por classificar_temas.py)

ROTULO = {"tema": "Tema", "macro_tema": "Macro-tema"}
COR = {"PPG-CCMC": "#0072B2", "PIPGES": "#E69F00", "PPG-Mat": "#009E73"}
ESCALA = [[0, "#e5e7eb"], [0.00001, "#e5e7eb"], [0.0000101, "#c6dbef"], [0.25, "#6baed6"],
          [0.5, "#3182bd"], [0.75, "#08519c"], [1, "#08306b"]]     # zero = cinza; resto, azul claro -> escuro


In [ ]:
# Dados: cada banca com seu tema, e o perfil de cada orientador (nº de bancas e concentração no tema principal)
eventos = pd.read_csv(CAMINHO)[["id", "orientador"]]
bancas = pd.read_csv(ARQ_TEMAS).merge(eventos, on="id")
bancas = bancas[bancas.tema != "Outros"]

PROGRAMAS = [p for p in COR if p in set(bancas.programa)]
MAX_BANCAS = int(bancas.orientador.value_counts().max())


def curto(s, n=22):
    return s if len(s) <= n else s[: n - 1] + "…"


def perfis(programa, minimo, nivel):
    """Uma linha por orientador (n, tema principal, concentração, programa) e a tabela orientador x tema com o nº de bancas."""
    b = bancas if programa == "Todos" else bancas[bancas.programa == programa]
    conta = pd.crosstab(b.orientador, b[nivel])
    n = conta.sum(axis=1)
    conta, n = conta[n >= minimo], n[n >= minimo]
    perfil = pd.DataFrame({"n": n, "tema_principal": conta.idxmax(axis=1), "concentracao": conta.div(n, axis=0).max(axis=1)})
    perfil["programa"] = b.groupby("orientador").programa.agg(lambda s: s.value_counts().index[0]).reindex(perfil.index)
    perfil = perfil.sort_values(["concentracao", "n"], ascending=False)       # mais especialistas primeiro
    return perfil, conta.loc[perfil.index]


In [ ]:
# Dispersão (seleção por caixa diagonal) ligada ao mapa de calor: o mapa mostra só os orientadores selecionados
def sem_acento(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()


def casa(nome, busca):
    return busca.strip() != "" and sem_acento(busca.strip()) in sem_acento(nome)


def ordenar(perfil, ordem):
    return {
        "esp": perfil.sort_values(["concentracao", "n"], ascending=False),
        "ger": perfil.sort_values(["concentracao", "n"], ascending=[True, False]),
        "n": perfil.sort_values("n", ascending=False),
        "nome": perfil.sort_index(),
    }[ordem]


def painel_ligado():
    estado = {"perfil": None, "conta": None, "nome_do_ponto": [], "selecao": None}

    def caixa():     # os cinco controles dividem a largura do gráfico em partes iguais
        return dict(layout=widgets.Layout(flex="1 1 0", min_width="0", margin="0 24px 6px 0"), style={"description_width": "initial"})

    dd_prog = widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa", **caixa())
    dd_nivel = widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível", **caixa())
    sl_min = widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas", **caixa())
    tx_busca = widgets.Text(placeholder="Digite um nome", description="Buscar", **caixa())
    dd_ordem = widgets.Dropdown(options=[("Mais especialistas", "esp"), ("Mais generalistas", "ger"),
                                         ("Nº de bancas", "n"), ("Nome", "nome")], description="Ordem", **caixa())

    # --- dispersão: um único trace selecionável (a seleção do plotly funciona por trace), mais traces só para a legenda
    # e um trace para circular o resultado da busca
    disp = go.FigureWidget()
    disp.add_trace(go.Scatter(mode="markers", hoverinfo="text", showlegend=False,
                              marker=dict(size=10, line=dict(color="white", width=1)),
                              selected=dict(marker=dict(opacity=1, size=12)),
                              unselected=dict(marker=dict(opacity=0.15))))
    for prog in PROGRAMAS:
        disp.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=prog, hoverinfo="skip",
                                  marker=dict(size=10, color=COR[prog])))
    disp.add_trace(go.Scatter(mode="markers+text", showlegend=False, hoverinfo="skip", textposition="top center",
                              marker=dict(size=20, color="rgba(0,0,0,0)", line=dict(color="black", width=2)),
                              unselected=dict(marker=dict(opacity=1), textfont=dict(color="black"))))
    i_busca = len(disp.data) - 1
    disp.update_layout(height=470, margin=dict(l=70, r=20, t=30, b=45), plot_bgcolor="#fafafa",
                       dragmode="select", selectdirection="d",
                       legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom", itemclick=False, itemdoubleclick=False),
                       title=dict(text="", font=dict(size=13), x=0))
    disp._config = {"responsive": True, "displayModeBar": False}      # largura acompanha a saída; sem a barra do plotly (o arrasto já seleciona)
    disp.update_xaxes(title_text="Nº de bancas do orientador", dtick=1, zeroline=False)
    disp.update_yaxes(title_text="Concentração no tema principal (% das bancas)", range=[0, 1.04], tickformat=".0%", zeroline=False)

    # --- mapa de calor: começa vazio e é preenchido por atualizar_mapa
    mapa = go.FigureWidget(go.Heatmap(hoverinfo="text", colorscale=ESCALA, zmin=0, zmax=1,
                                      colorbar=dict(title="% das bancas", tickformat=".0%", x=1.01, xanchor="left",
                                                    y=1, yanchor="top", lenmode="pixels", len=260, thickness=14)))
    mapa.update_xaxes(tickangle=-55, side="top", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    mapa.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    mapa.update_layout(margin=dict(l=130, r=20, b=10), plot_bgcolor="white", dragmode=False)
    mapa._config = {"responsive": True, "displayModeBar": False}

    def atualizar_mapa(nomes):
        """Redesenha o mapa de calor só com os orientadores em `nomes` (todos, se for None), na ordem escolhida."""
        estado["selecao"] = list(nomes) if nomes else None
        perfil, conta, rot = estado["perfil"], estado["conta"], ROTULO[dd_nivel.value]
        base = perfil.loc[[o for o in perfil.index if o in set(nomes)]] if nomes else perfil
        ordem = list(ordenar(base, dd_ordem.value).index)
        if nomes:
            texto_info = f"<b>{len(ordem)}</b> de {len(perfil)} orientadores selecionados"
        else:
            texto_info = f"Nenhuma seleção, o mapa mostra os {len(perfil)} orientadores"
        achados = [i for i, o in enumerate(ordem) if casa(o, tx_busca.value)]
        if tx_busca.value.strip():
            texto_info += f" · {len(achados)} encontrado(s) no mapa (contorno preto)" if achados else " · busca sem resultado no mapa"
        disp.layout.title.text = texto_info

        sub = conta.loc[ordem]
        sub = sub.loc[:, sub.sum() > 0]                                    # só os temas que alguém usou
        pct = sub.div(perfil.n[ordem], axis=0)
        texto = [[f"<b>{o}</b> ({perfil.n[o]} bancas)<br>{rot}: {t}<br>"
                  + (f"Concentração: {pct.loc[o, t]:.0%} das bancas ({sub.loc[o, t]}/{perfil.n[o]})" if sub.loc[o, t] else "Nenhuma banca nesse tema")
                  for o in sub.index] for t in sub.columns]
        nomes_curtos = [curto(o, 24) for o in sub.index]                  # nome completo só no hover, para o topo ocupar menos espaço
        topo = int(0.82 * 5.5 * max((len(o) for o in nomes_curtos), default=0)) + 30
        contornos = [dict(type="rect", xref="x", yref="y", x0=i - 0.5, x1=i + 0.5, y0=-0.5, y1=len(sub.columns) - 0.5,
                          line=dict(color="black", width=2.5), layer="above") for i in achados]
        with mapa.batch_update():
            mapa.data[0].x, mapa.data[0].y, mapa.data[0].z, mapa.data[0].text = list(sub.index), list(sub.columns), pct.values.T, texto
            mapa.layout.xaxis.tickvals, mapa.layout.xaxis.ticktext = list(sub.index), nomes_curtos
            mapa.layout.yaxis.tickvals = list(sub.columns)
            mapa.layout.yaxis.ticktext = [curto(t) for t in sub.columns]
            mapa.layout.height = max(340, topo + 26 * len(sub.columns) + 10)
            mapa.layout.margin.t = topo
            mapa.layout.shapes = contornos

    def atualizar_busca(*_):
        """Circula na dispersão os orientadores que casam com a busca e contorna a coluna deles no mapa."""
        perfil = estado["perfil"]
        achados = perfil[[casa(o, tx_busca.value) for o in perfil.index]]
        x = [estado["x"][o] for o in achados.index]
        with disp.batch_update():
            t = disp.data[i_busca]
            t.x, t.y, t.text = x, list(achados.concentracao), list(achados.index)
        atualizar_mapa(estado["selecao"])

    def atualizar_dados(*_):
        """Recalcula tudo quando muda programa, nível ou mínimo de bancas (e limpa a seleção)."""
        nivel = dd_nivel.value
        rot = ROTULO[nivel]
        perfil, conta = perfis(dd_prog.value, sl_min.value, nivel)
        estado.update(perfil=perfil, conta=conta, nome_do_ponto=list(perfil.index), selecao=None)
        estado["x"] = {o: n + ((zlib.crc32(o.encode()) % 1000) / 1000 - 0.5) * 0.36 for o, n in zip(perfil.index, perfil.n)}   # deslocamento p/ não sobrepor
        texto = [f"<b>{o}</b> ({r.n} bancas)<br>{rot} principal: {r.tema_principal}<br>"
                 f"Concentração: {r.concentracao:.0%} ({round(r.concentracao * r.n)}/{r.n})" for o, r in perfil.iterrows()]
        a, b = (perfil.n.min(), perfil.n.max()) if len(perfil) else (1, MAX_BANCAS)
        with disp.batch_update():
            t = disp.data[0]
            t.x, t.y, t.text = [estado["x"][o] for o in perfil.index], list(perfil.concentracao), texto
            t.marker.color = [COR[p] for p in perfil.programa]
            t.selectedpoints = None
            disp.layout.xaxis.range = [a - 0.7, b + 0.7]
            disp.layout.yaxis.title.text = f"Concentração no {rot.lower()} principal (% das bancas)"
        atualizar_busca()

    def ao_selecionar(trace, points, selector):
        atualizar_mapa([estado["nome_do_ponto"][i] for i in points.point_inds])

    disp.data[0].on_selection(ao_selecionar)
    disp.data[0].on_deselect(lambda trace, points: atualizar_mapa(None))      # duplo clique no gráfico
    for w in (dd_prog, dd_nivel, sl_min):
        w.observe(atualizar_dados, names="value")
    tx_busca.observe(atualizar_busca, names="value")
    dd_ordem.observe(lambda *_: atualizar_mapa(estado["selecao"]), names="value")

    atualizar_dados()
    dd_ordem.layout.margin = "0 0 6px 0"                                          # o último não precisa de espaço à direita
    controles = widgets.HBox([dd_prog, dd_nivel, sl_min, tx_busca, dd_ordem], layout=widgets.Layout(margin="24px 70px 0 70px"))     # espaço acima e o mesmo espaço nas duas laterais
    display(widgets.VBox([controles, disp, mapa], layout=widgets.Layout(width="100%")))
    return disp, mapa, atualizar_mapa, estado


disp, mapa, atualizar_mapa, estado = painel_ligado()


## Acessibilidade

Os programas usam a paleta Okabe-Ito, a mesma das outras questões, pensada para daltonismo. O mapa usa uma escala de um só matiz, do azul claro ao escuro, e as células sem bancas ficam em cinza.

A seleção não depende de cor. Os pontos não selecionados ficam apagados, e a busca circula o ponto com contorno preto e contorna a coluna dele no mapa. O nome, o número de bancas e a concentração também aparecem em texto no hover.

## O que dá para ver

Caixas de exemplo, desenhadas à mão, então os números variam um pouco.

- Muitas bancas e concentração alta (8 ou mais bancas, 80% ou mais): cerca de 9 orientadores, todos do PPG-CCMC. O mapa mostra em quais temas eles se concentram.
- Muitas bancas e concentração baixa (8 ou mais bancas, 35% ou menos): 3 orientadores, também do PPG-CCMC, que atuam em cerca de 6 temas cada.
- Poucas bancas e concentração alta (4 ou 5 bancas, 80% ou mais): 12 orientadores, a maioria do PPG-Mat e do PIPGES. Com tão poucas bancas, a concentração alta pode ser coincidência.

Aprendizado de Máquina e Mineração de Dados é o tema principal mais comum entre os especialistas, entre os orientadores com 15 ou mais bancas e na faixa de concentração intermediária (entre 40% e 60%). Para o aluno, a seleção permite escolher um perfil de orientador, como quem tem muitas bancas, e ver em quais temas ele atua.

## Limitações

- Uma caixa grande traz de volta um mapa com muitas colunas apertadas.
- Os pontos caem em posições discretas, porque o número de bancas é inteiro e a concentração é uma fração. O pequeno deslocamento horizontal ajuda, mas ainda pode esconder pontos.
- Valem as limitações da Questão 3 da primeira entrega. O tema vem de um modelo de linguagem sem taxa de acerto, cada banca tem um tema só, MECAI e PROFMAT ficam de fora, e poucas bancas facilitam uma concentração alta por acaso.